# Sanity checks and confounded features

**Weeks:** 34 · **Core question:** Can a heatmap highlight an acquisition artifact instead of biology?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

Explanations can be sensitive to artifacts correlated with labels. We intentionally plant a corner marker in positive training images. We then compare predictions and gradient saliency when the artifact is absent. Do **not** interpret synthetic results as evidence about any real model or hospital.

**Key vocabulary:** spurious correlation, model randomization, data shortcut, domain shift.

**Learning goal:** Document two possible failures for a seemingly meaningful visual explanation.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
import torch
from diffcam_edu import make_cell_images
from diffcam_edu.models import fit_cnn,gradcam
from diffcam_edu.math import normalize_map
x,y,m=make_cell_images(260,seed=50,artifact_strength=.7)
model,history=fit_cnn(x[:200],y[:200],epochs=7,seed=5)
clean,clean_y,_=make_cell_images(100,seed=51,artifact_strength=0.)
with torch.no_grad():
    acc_train=np.mean(model(torch.tensor(x[200:])[:,None]).argmax(1).numpy()==y[200:])
    acc_clean=np.mean(model(torch.tensor(clean)[:,None]).argmax(1).numpy()==clean_y)
print('Artifact-present test accuracy:',acc_train,'Artifact-free test accuracy:',acc_clean)

### Experiment 2 of 3

In [ ]:
i=int(np.flatnonzero(y[200:]==1)[0]+200)
heat,_=gradcam(model,x[i],1)
fig,ax=plt.subplots(1,2,figsize=(7,3))
ax[0].imshow(x[i],cmap='gray');ax[0].set_title('Confounded input')
ax[1].imshow(normalize_map(heat),cmap='inferno');ax[1].set_title('Grad-CAM')
for a in ax:a.axis('off')
plt.show()

### Experiment 3 of 3

In [ ]:
from diffcam_edu.models import build_cnn
random_model=build_cnn(seed=678)
random_heat,_=gradcam(random_model,x[i],1)
print('Original vs random-weight map mean abs difference:',np.abs(normalize_map(heat)-normalize_map(random_heat)).mean())

## Exercises — complete independently
1. Compare DiffCAM maps using clean references versus artifact-correlated references.
2. Why should saliency sanity checks test both model sensitivity and data confounding?

## Check your understanding

**Exit ticket:** Document two possible failures for a seemingly meaningful visual explanation.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)